In [1]:
import pandas as pd
import numpy as np
import re
from math import log2

HUMAN_MATRIX_CSV = "human_matrix_answered_9users.csv"
LLM_CHOICES_CSV = "all_llm_choices.csv"  # change to informed file if needed

OUT_ALIGNMENT = "llm_human_alignment.csv"

OPTIONS = list("ABCDE")

# If cells contain repeats like D|D:
# True = count both repeated presentations
# False = count each participant at most once per question
COUNT_REPEATS_IN_CELL = True


def normalize_to_letter(x):
    """Map values like 'option_B', 'B', 'b', '2' -> A-E or NaN."""
    if pd.isna(x):
        return np.nan

    s = str(x).strip().upper()

    m = re.search(r"\b([A-E])\b", s)
    if m:
        return m.group(1)

    if s.isdigit():
        mapping = {1: "A", 2: "B", 3: "C", 4: "D", 5: "E"}
        return mapping.get(int(s), np.nan)

    return np.nan


def letters_from_cell(cell, count_repeats=True):
    """Handle '', 'D', 'option_D', or repeated cells like 'D|D'."""
    if pd.isna(cell):
        return []

    s = str(cell).strip()
    if not s:
        return []

    parts = [p.strip() for p in s.split("|")]
    letters = [normalize_to_letter(p) for p in parts]
    letters = [l for l in letters if l in OPTIONS]

    if not count_repeats:
        letters = sorted(set(letters))

    return letters


def normalize_counts(counts):
    arr = np.array([counts.get(o, 0) for o in OPTIONS], dtype=float)
    total = arr.sum()
    if total == 0:
        return np.zeros_like(arr)
    return arr / total


def js_distance(p, q):
    """Jensen-Shannon distance, lower = closer."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if p.sum() == 0 or q.sum() == 0:
        return np.nan

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    return np.sqrt(0.5 * kl(p, m) + 0.5 * kl(q, m))


def cosine_similarity(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    denom = np.linalg.norm(p) * np.linalg.norm(q)
    if denom == 0:
        return np.nan

    return float(np.dot(p, q) / denom)


# -----------------------
# 1) Load human matrix
# -----------------------
hm = pd.read_csv(HUMAN_MATRIX_CSV)

if "question_id" in hm.columns:
    hm = hm.set_index("question_id")
elif "Unnamed: 0" in hm.columns:
    hm = hm.rename(columns={"Unnamed: 0": "question_id"}).set_index("question_id")
else:
    hm = hm.set_index(hm.columns[0])
    hm.index.name = "question_id"

hm.index = hm.index.astype(int)
question_ids = sorted(hm.index.tolist())

meta_cols = [c for c in hm.columns if c.startswith("n_") or c.startswith("count_")]
user_cols = [c for c in hm.columns if c not in meta_cols]

# Human counts per question + human majority option(s)
human_by_question = {}

for qid, row in hm.iterrows():
    counts = {o: 0 for o in OPTIONS}

    for col in user_cols:
        for letter in letters_from_cell(row[col], count_repeats=COUNT_REPEATS_IN_CELL):
            counts[letter] += 1

    max_count = max(counts.values())
    majority = [o for o, c in counts.items() if c == max_count and c > 0]

    human_by_question[qid] = {
        "counts": counts,
        "majority": majority,
        "n_human_votes": sum(counts.values()),
    }

# Overall human distribution across this question set
human_total_counts = {o: 0 for o in OPTIONS}
for q in human_by_question.values():
    for o in OPTIONS:
        human_total_counts[o] += q["counts"][o]

human_dist = normalize_counts(human_total_counts)

print("Human total counts:", human_total_counts)
print("Human distribution:", dict(zip(OPTIONS, human_dist.round(4))))


# -----------------------
# 2) Load LLM choices
# -----------------------
llm = pd.read_csv(LLM_CHOICES_CSV)

qid_col = None
for candidate in ["question_id", "number", "Question ID", "qid"]:
    if candidate in llm.columns:
        qid_col = candidate
        break

if qid_col is None:
    raise ValueError("Could not find question id column in LLM file.")

llm[qid_col] = pd.to_numeric(llm[qid_col], errors="coerce").astype("Int64")
llm = llm[llm[qid_col].isin(question_ids)].copy()

model_cols = [c for c in llm.columns if c != qid_col]

results = []

for model in model_cols:
    model_counts = {o: 0 for o in OPTIONS}
    majority_matches = 0
    answered = 0
    comparable = 0

    for _, row in llm.iterrows():
        qid = int(row[qid_col])
        choice = normalize_to_letter(row[model])

        if choice not in OPTIONS:
            continue

        answered += 1
        model_counts[choice] += 1

        human_majority = human_by_question[qid]["majority"]
        if human_majority:
            comparable += 1
            if choice in human_majority:
                majority_matches += 1

    model_dist = normalize_counts(model_counts)

    l1_distance = float(np.abs(model_dist - human_dist).sum())
    js = js_distance(model_dist, human_dist)
    cosine = cosine_similarity(model_dist, human_dist)

    results.append({
        "model": model,
        "answered_questions": answered,
        "comparable_questions": comparable,
        "human_majority_matches": majority_matches,
        "human_majority_agreement": majority_matches / comparable if comparable else np.nan,
        "A_count": model_counts["A"],
        "B_count": model_counts["B"],
        "C_count": model_counts["C"],
        "D_count": model_counts["D"],
        "E_count": model_counts["E"],
        "A_pct": model_dist[0] * 100,
        "B_pct": model_dist[1] * 100,
        "C_pct": model_dist[2] * 100,
        "D_pct": model_dist[3] * 100,
        "E_pct": model_dist[4] * 100,
        "L1_distance_to_human_distribution": l1_distance,
        "JS_distance_to_human_distribution": js,
        "cosine_similarity_to_human_distribution": cosine,
    })

out = pd.DataFrame(results)

# Higher majority agreement is better; lower JS/L1 is better; higher cosine is better.
out = out.sort_values(
    by=["human_majority_agreement", "JS_distance_to_human_distribution"],
    ascending=[False, True],
)

out.to_csv(OUT_ALIGNMENT, index=False)

print("\nModel alignment with human judgments:")
print(out[[
    "model",
    "human_majority_agreement",
    "human_majority_matches",
    "comparable_questions",
    "JS_distance_to_human_distribution",
    "L1_distance_to_human_distribution",
    "cosine_similarity_to_human_distribution",
]])

print("\nBest by human-majority agreement:")
print(out.iloc[0][["model", "human_majority_agreement", "human_majority_matches", "comparable_questions"]])

best_by_js = out.sort_values("JS_distance_to_human_distribution").iloc[0]
print("\nBest by distribution similarity / JS distance:")
print(best_by_js[["model", "JS_distance_to_human_distribution", "L1_distance_to_human_distribution", "cosine_similarity_to_human_distribution"]])

print("\nSaved:", OUT_ALIGNMENT)

Human total counts: {'A': 23, 'B': 26, 'C': 45, 'D': 68, 'E': 18}
Human distribution: {'A': np.float64(0.1278), 'B': np.float64(0.1444), 'C': np.float64(0.25), 'D': np.float64(0.3778), 'E': np.float64(0.1)}

Model alignment with human judgments:
               model  human_majority_agreement  human_majority_matches  \
7   qwen3maxthinking                  0.555556                      10   
5            gpt5pro                  0.555556                      10   
0     claude35sonnet                  0.555556                      10   
4              gpt52                  0.555556                      10   
3              gpt41                  0.500000                       9   
6  llama323binstruct                  0.500000                       9   
1     claudesonnet45                  0.444444                       8   
2        deepseekv32                  0.388889                       7   

   comparable_questions  JS_distance_to_human_distribution  \
7                    18  